In [51]:
import cv2
import os
import shutil
import tensorflow as tf
import numpy  as np
from os import path
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Add, Dense, GlobalAveragePooling2D, MaxPooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Input
from tensorflow.keras.callbacks import ReduceLROnPlateau
import matplotlib.pyplot as plt

In [34]:
Data_path = r"D:\BTS\2DAI\PFE\Projects\PFE__Model"

X_train = np.load(path.join(Data_path, 'trainX.npy'))
Y_train = np.load(path.join(Data_path, 'trainY.npy'))

X_test = np.load(path.join(Data_path, 'testX.npy'))
Y_test = np.load(path.join(Data_path, 'testY.npy'))

In [35]:
## data augmentation 
datagen = ImageDataGenerator( rotation_range=30,    width_shift_range=0.2,  height_shift_range=0.2, shear_range=0.2, 
zoom_range=0.2, horizontal_flip=True,)

In [36]:

#transfert the model
base_Model= InceptionV3(weights="imagenet", include_top=False, input_tensor=Input(shape=(224, 224, 3))) 

In [37]:
#change the head
head_network = base_Model.output
head_network = GlobalAveragePooling2D()(head_network)
head_network = Dense(250, activation = "relu")(head_network)
head_network = Dropout(0.5)(head_network)
head_network = Dense(3, activation = 'softmax')(head_network)

model = Model(inputs = base_Model.input, outputs = head_network)

In [38]:
#Fine-Tuning
base_Model.trainable = True
for layer in base_Model.layers[:102]:
    layer.trainable = False 

In [39]:
# initial learning rate- epochs -batch size
INIT_LR = 1e-4
Epochs =15
#compling
opt = Adam(lr=INIT_LR, decay=INIT_LR / Epochs)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["acc"])

In [41]:
#stoping when overfitting
stoping= EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=7)
#saving the best version
B_save = ModelCheckpoint('PFE__Model/X-rays_model.h5', save_best_only=True, monitor='val_acc', mode='max')
callbacks = [B_save, stoping]

In [ ]:

print("start training")
new_head = model.fit(
    datagen.flow(X_train, Y_train, batch_size=32),
    steps_per_epoch=len(X_train) // 32,
    validation_data=(X_test, Y_test), 
    epochs=100,
    callbacks=callbacks,
    shuffle=True #melanger data
)

start training
Epoch 1/100
75/75 [==============================] - 30s 297ms/step - loss: 0.6130 - acc: 0.7275 - val_loss: 0.8197 - val_acc: 0.5500
Epoch 2/100
75/75 [==============================] - 19s 254ms/step - loss: 0.4121 - acc: 0.8271 - val_loss: 0.6733 - val_acc: 0.6983
Epoch 3/100
75/75 [==============================] - 20s 255ms/step - loss: 0.3360 - acc: 0.8608 - val_loss: 0.4149 - val_acc: 0.8283
Epoch 4/100
75/75 [==============================] - 19s 257ms/step - loss: 0.3183 - acc: 0.8600 - val_loss: 0.3391 - val_acc: 0.8333
Epoch 5/100
75/75 [==============================] - 21s 282ms/step - loss: 0.2787 - acc: 0.8829 - val_loss: 0.3490 - val_acc: 0.8500
Epoch 6/100
75/75 [==============================] - 20s 260ms/step - loss: 0.3039 - acc: 0.8758 - val_loss: 0.2292 - val_acc: 0.9067
Epoch 7/100
75/75 [==============================] - 20s 259ms/step - loss: 0.2752 - acc: 0.8892 - val_loss: 0.1690 - val_acc: 0.9300
Epoch 8/100
75/75 [============================

In [44]:
# stopin 50 layer
for layer in base_Model.layers[:50]:
    layer.trainable = False

In [ ]:
stoping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
B_save = ModelCheckpoint('X-rays_best_model.h5',save_best_only=True, monitor='val_acc', mode='max')

In [46]:
m = model.fit_generator(
    datagen.flow(X_train, Y_train, batch_size=32),
    steps_per_epoch=len(X_train) // 32,
    validation_data=(X_test, Y_test),
    epochs=100, 
    callbacks=[B_save, stoping]
)

C:\Users\MSI\AppData\Local\Temp\ipykernel_28268\2641758536.py:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  m = model.fit_generator(


Epoch 1/100
75/75 [==============================] - 21s 277ms/step - loss: 0.1562 - acc: 0.9367 - val_loss: 0.1180 - val_acc: 0.9550
Epoch 2/100
75/75 [==============================] - 19s 251ms/step - loss: 0.1290 - acc: 0.9517 - val_loss: 0.3138 - val_acc: 0.8867
Epoch 3/100
75/75 [==============================] - 21s 275ms/step - loss: 0.1663 - acc: 0.9354 - val_loss: 0.1039 - val_acc: 0.9600
Epoch 4/100
75/75 [==============================] - 19s 255ms/step - loss: 0.1486 - acc: 0.9454 - val_loss: 0.1631 - val_acc: 0.9450
Epoch 5/100
75/75 [==============================] - 19s 253ms/step - loss: 0.1449 - acc: 0.9425 - val_loss: 0.1782 - val_acc: 0.9117
Epoch 6/100
75/75 [==============================] - 19s 253ms/step - loss: 0.1318 - acc: 0.9508 - val_loss: 0.1864 - val_acc: 0.9183
Epoch 7/100
75/75 [==============================] - 19s 252ms/step - loss: 0.1578 - acc: 0.9413 - val_loss: 0.2393 - val_acc: 0.8967
Epoch 8/100
75/75 [==============================] - 19s 252ms

In [ ]:
#Learning Rate
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
callbacks = [B_save, stoping, reduce_lr]

In [ ]:

m = model.fit_generator(
    datagen.flow(X_train, Y_train, batch_size=32),
    steps_per_epoch=len(X_train) // 32,
    validation_data=(X_test, Y_test),
    epochs=100, 
    callbacks=callbacks
)

C:\Users\MSI\AppData\Local\Temp\ipykernel_28268\3045497088.py:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  m = model.fit_generator(


Epoch 1/100
75/75 [==============================] - 22s 280ms/step - loss: 0.1466 - acc: 0.9433 - val_loss: 0.1218 - val_acc: 0.9517 - lr: 1.0000e-04
Epoch 2/100
75/75 [==============================] - 20s 258ms/step - loss: 0.1290 - acc: 0.9483 - val_loss: 0.1503 - val_acc: 0.9317 - lr: 1.0000e-04
Epoch 3/100
75/75 [==============================] - 19s 250ms/step - loss: 0.1257 - acc: 0.9563 - val_loss: 0.2089 - val_acc: 0.9167 - lr: 1.0000e-04
Epoch 4/100
75/75 [==============================] - ETA: 0s - loss: 0.1394 - acc: 0.9442
Epoch 4: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
75/75 [==============================] - 19s 252ms/step - loss: 0.1394 - acc: 0.9442 - val_loss: 0.1692 - val_acc: 0.9233 - lr: 1.0000e-04
Epoch 5/100
75/75 [==============================] - 21s 280ms/step - loss: 0.1082 - acc: 0.9588 - val_loss: 0.0910 - val_acc: 0.9633 - lr: 5.0000e-05
Epoch 6/100
75/75 [==============================] - 25s 330ms/step - loss: 0.1001 - acc: 0